In [ ]:
#!/usr/bin/env python3
import os
import random
import pickle
import argparse
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple

try:
    import psutil
except Exception:
    psutil = None

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

import lightgbm as lgb

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)

METRICS_LOG = "pipeline_metrics.jsonl"
CONSOLIDATED_BEST = "pipe_best_clis_all.txt"
CONSOLIDATED_ALT = "pipe_alt_clis_all.txt"
CONSOLIDATED_PF = "pipeline_configs.npy"

if os.path.exists(METRICS_LOG):
    os.remove(METRICS_LOG)

def append_metrics(row: dict):
    with open(METRICS_LOG, "a") as f:
        f.write(json.dumps(row, default=lambda x: None) + "\n")

df_real = pd.read_excel("pipe_data.xlsx")

def generate_groupings(num_stages: int):
    ids = list(range(1, num_stages + 1))
    plans = [[[s] for s in ids]]
    for i in range(num_stages - 1):
        g = [[s] for s in ids]
        g[i] = [ids[i], ids[i + 1]]
        g.pop(i + 1)
        if g not in plans:
            plans.append(g)
        if len(plans) >= 5:
            break
    return plans

stage_list = sorted(df_real["num_stages"].unique())
grouping_plans = {s: generate_groupings(s) for s in stage_list}

queue_sizes = [1, 10, 100, 1000, 10000, 100000, 1000000]
threads_per_stage = list(range(1, 9))
core_combos = [(b, l) for b in range(5) for l in range(5) if b + l > 0]

freq_little = [f * 1000 for f in range(200, 1501, 100)]
freq_big = [f * 1000 for f in range(200, 2001, 100)]
freq_samples = []
while len(freq_samples) < 500:
    lf = random.choice(freq_little)
    bf = random.choice(freq_big)
    freq_samples.append([lf] * 4 + [bf] * 4)

df = df_real.dropna(subset=["energy", "time"]).copy()
for i in range(1, 11):
    col = f"workload_size_stage{i}"
    df[col] = df.get(col, 0.0).fillna(0.0).astype(float)

df["total_cores_on"] = df["big_cores_on"] + df["little_cores_on"]
df["imbalance"] = abs(df["big_cores_on"] - df["little_cores_on"])
df["ratio_big_to_little"] = df["big_cores_on"] / (df["little_cores_on"] + 1e-6)

cpu_cols = [c for c in df.columns if c.startswith("cpu") and "_freq" in c]
little = [c for c in cpu_cols if int(''.join(filter(str.isdigit, c))) < 4]
big = [c for c in cpu_cols if int(''.join(filter(str.isdigit, c))) >= 4]
df["avg_little_freq"] = df[little].mean(axis=1) if little else 1_500_000
df["avg_big_freq"] = df[big].mean(axis=1) if big else 2_000_000

desc_map = {}
for s, plans in grouping_plans.items():
    for gid, groups in enumerate(plans, start=1):
        desc = ",".join("+".join(f"s{i}" for i in g) for g in groups)
        desc_map (df["little_cores_on"] + 1e-6)

cpu_cols = [c for c in df.columns if c.startswith("cpu") and "_freq" in c]
little = [c for c in cpu_cols if int(''.join(filter(str.isdigit, c))) < 4]
big = [c for c in cpu_cols if int(''.join(filter(str.isdigit, c))) >= 4]
df["avg_little_freq"] = df[little].mean(axis=1) if little else 1_500_000
df["avg_big_freq"] = df[big].mean(axis=1) if big else 2_000_000

desc_map = {}
for s, plans in grouping_plans.items():
    for gid, groups in enumerate(plans, start=1):
        desc = ",".join("+".join(f"s{i}" for i in g[(s, gid)] = desc

df["group_description"] = df.apply(
    lambda r: desc_map.get((r.num_stages, int(r.grouping)), f"g{int(r.grouping)}"),
    axis=1) for g in groups)
        desc_map[(s, gid)] = desc

df["group_description"] = df.apply(
    lambda r: desc_map.get((r.num_stages, int(r.grouping)), f"g{int(r.grouping)}"),
    axis=1
)

cat_cols = [c for c in ["workload_type", "core_binding_type", "group_description", "grouping"] if c in df]
num_cols
)

cat_cols = [c for c in ["workload_type", "core_binding_type", "group_description", "grouping"] if c in df]
num_cols = (
    ["num_stages", "big_cores_on", "little_cores_on",
     "total_cores_on", "imbalance", "ratio_big_to_little",
     "avg_little_freq", "avg_big_freq"]
    = (
    ["num_stages", "big_cores_on", "little_cores_on",
     "total_cores_on", "imbalance", "ratio_big_to_little",
     "avg_little_freq", "avg_big_freq"]
    + [f"workload_size_stage{i}" for i in range(1, 11)]
    + [f"workload_size_stage{i}" for i in range(1, 11)]
    + [f"stage{i}_threads" for i in range(1, 11)]
    + [f"queue{i}_size" for i in range(1, 11)]
)
 + [f"stage{i}_threads" for i in range(1, 11)]
    + [f"queue{i}_size" for i in range(1, 11)]
)
num_cols = [c for c in num_cols if c in df]

X = pd.concat([df[cat_cols].astype(str), df[num_cols]], axis=1)
y = df[["energy", "time"]]

preproc = Columnnum_cols = [c for c in num_cols if c in df]

X = pd.concat([df[cat_cols].astype(str), df[num_cols]], axis=1)
y = df[["energy", "time"]]

preproc = ColumnTransformer([
    ("num", SimpleImputer(strategy="mean"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

lgb_params = {"n_estimators": 200Transformer([
    ("num", SimpleImputer(strategy="mean"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

lgb_params = {"n_estimators": 200, "random_state": SEED, "n_jobs": -1}
model_pipe =, "random_state": SEED, "n_jobs": -1}
model_pipe = Pipeline([
    ("prep", preproc),
    ("reg", MultiOutputRegressor(lgb.LGBMRegressor(**lgb_params)))
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size Pipeline([
    ("prep", preproc),
    ("reg", MultiOutputRegressor(lgb.LGBMRegressor(**lgb_params)))
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.15, random_state=0.15, random_state=SEED)

def fit_with_metrics(pipe: Pipeline, X_train: pd.DataFrame, y_train: pd.DataFrame) -> Tuple[Pipeline, float, int]:
    start = time.perf_counter()
=SEED)

def fit_with_metrics(pipe: Pipeline, X_train: pd.DataFrame, y_train: pd.DataFrame) -> Tuple[Pipeline, float, int]:
    start = time.perf_counter()
    peak_mem = -1
    if psutil is None:
        pipe.fit(X_train, y_train)
        train_time = time.perf_counter() - start
        return pipe, train_time, -1
    proc = psutil.Process()
    import threading    peak_mem = -1
    if psutil is None:
        pipe.fit(X_train, y_train)
        train_time = time.perf_counter() - start
        return pipe, train_time, -1
    proc = psutil.Process()
    import threading
    done = threading.Event()
    mem_samples = []
    def sampler():
        while not done.is_set():
            try:
                mem_samples.append(proc.memory_info().rss)
            except Exception:
                pass
           
    done = threading.Event()
    mem_samples = []
    def sampler():
        while not done.is_set():
            try:
                mem_samples.append(proc.memory_info().rss)
            except Exception:
                pass
            time.sleep(0.01 time.sleep(0.01)
    thread = threading.Thread(target=sampler, daemon=True)
    thread.start()
    try:
        pipe.fit(X_train, y_train)
    finally:
        done.set()
        thread.join(timeout=1.0)
    train_time = time.perf_counter() - start
    peak_mem = int(max(mem_samples) if mem_samples else -1)
    return pipe, train_time, peak_mem

model_pipe, train_time_s, peak_mem_bytes = fit_with_metrics(model_pipe, Xtr, ytr)

)
    thread = threading.Thread(target=sampler, daemon=True)
    thread.start()
    try:
        pipe.fit(X_train, y_train)
    finally:
        done.set()
        thread.join(timeout=1.0)
    train_time = time.perf_counter() - start
    peak_mem = int(max(mem_samples) if mem_samples else -1)
    return pipe, train_time, peak_mem

model_pipe, train_time_s, peak_mem_bytes = fit_with_metrics(model_pipe, Xtr, ytr)

preds = model_pipe.predict(Xte)
rmse_e = np.sqrt(((preds[:, 0] - yte.energy) ** 2).mean())
rmse_t = np.sqrt(((preds[:, 1] - yte.timepreds = model_pipe.predict(Xte)
rmse_e = np.sqrt(((preds[:, 0] - yte.energy) ** 2).mean())
rmse_t = np.sqrt(((preds[:, 1] - yte.time) ** 2).mean())
) ** 2).mean())
mae_e = mean_absolute_error(yte["energy"], preds[:, 0])
mae_t = mean_absolute_error(yte["time"], preds[:, 1])
r2mae_e = mean_absolute_error(yte["energy"], preds[:, 0])
mae_t = mean_absolute_error(yte["time"], preds[:, 1])
r2_e = r2_score(yte["energy"], preds[:, 0])
r2_t = r2_score(yte["time"], preds[:, 1])

print(f"Surrogate trained → RMSE energy={rmse_e:.2f}, time={rmse_t:.2f}")
print(f"MAE           → energy={mae_e_e = r2_score(yte["energy"], preds[:, 0])
r2_t = r2_score(yte["time"], preds[:, 1])

print(f"Surrogate trained → RMSE energy={rmse_e:.2f}, time={rmse_t:.2f}")
print(f"MAE           → energy={mae_e:.2f}, time={mae_t:.2f}")
print(f"R2            → energy={r2_e:.3f}, time={r2_t:.3f}")
print(f"Train time (s): {train_time_s:.3f}, Peak:.2f}, time={mae_t:.2f}")
print(f"R2            → energy={r2_e:.3f}, time={r2_t:.3f}")
print(f"Train time (s): {train_time_s:.3f}, Peak mem (bytes): {peak_mem_bytes}")

baseline_mae_e = float(np.mean(np.abs(yte["energy"] - yte["energy"].mean())))
baseline_mae_t = float(np.mean(np.abs(yte["time"] - yte["time"].mean())))
yte_std_energy = float(yte["energy"]. mem (bytes): {peak_mem_bytes}")

baseline_mae_e = float(np.mean(np.abs(yte["energy"] - yte["energy"].mean())))
baseline_mae_t = float(np.mean(np.abs(yte["time"] - yte["time"].mean())))
yte_std_energy = float(yte["energy"].std())
yte_std_time = float(yte["time"].std())
r2_reliable = (yte_std_energy > 1e-6) and (yte_std_time > 1e-6)
print("baselinestd())
yte_std_time = float(yte["time"].std())
r2_reliable = (yte_std_energy > 1e-6) and (yte_std_time > 1e-6)
print("baseline MAE energy:", baseline_mae_e, "time:", baseline_mae_t)
print("yte std energy:", yte_std_energy, "time:", yte_std_time, "r2_reliable:", r2_reliable)

tmp_model_file = MAE energy:", baseline_mae_e, "time:", baseline_mae_t)
print("yte std energy:", yte_std_energy, "time:", yte_std_time, "r2_reliable:", r2_reliable)

tmp_model_file = "pipeline_model_tmp.pkl"
with open(tmp_model_file, "pipeline_model_tmp.pkl"
with open(tmp_model_file, "wb") as f:
    pickle.dump(model_pipe, f, protocol=pickle.HIGHEST_PROTOCOL)
model_size_bytes = os.path.getsize(tmp_model_file)
os.remove(tmp_model_file)
print(f"Model serialized size (bytes): {model_size_bytes}")

 "wb") as f:
    pickle.dump(model_pipe, f, protocol=pickle.HIGHEST_PROTOCOL)
model_size_bytes = os.path.getsize(tmp_model_file)
os.remove(tmp_model_file)
print(f"Model serialized size (bytes): {model_size_bytes}")

def predict_perf(pipe: Pipeline, Xc: pd.DataFrame, reps: int = 5) -> Tuple[float, float, np.ndarray]:
    _ = pipe.predictdef predict_perf(pipe: Pipeline, Xc: pd.DataFrame, reps: int = 5) -> Tuple[float, float, np.ndarray]:
    _ = pipe.predict(Xc.iloc[:min(10, len(Xc))])
    times = []
    preds_local = None
    for _ in range(reps):
        t0 = time.perf_counter()
        preds_local = pipe.predict(Xc)
        t1 = time.perf_counter()
        times.append((t1 - t0(Xc.iloc[:min(10, len(Xc))])
    times = []
    preds_local = None
    for _ in range(reps):
        t0 = time.perf_counter()
        preds_local = pipe.predict(Xc)
        t1 = time.perf_counter()
        times) / max(1, len(Xc)))
    return float(np.median(times)), float(np.std(times)), preds_local

pred_latency_per_sample_s, pred_latency_std_s, _ = predict.append((t1 - t0) / max(1, len(Xc)))
    return float(np.median(times)), float(np.std(times)), preds_local

pred_latency_per_sample_s, pred_latency_std_s, _ = predict_perf(model_pipe, Xte, reps=5)
print(f"Prediction latency per sample: {pred_latency_per_sample_s*1e3:.3f} ms (std {pred_latency_std_s*1e3:.3f} ms)")

_perf(model_pipe, Xte, reps=5)
print(f"Prediction latency per sample: {pred_latency_per_sample_s*1e3:.3f} ms (std {pred_latency_std_s*1e3:.3f} ms)")

pd.concat([Xte.reset_index(drop=True).iloc[:100],
           pd.DataFrame(preds, columns=["pred_energy", "pred_time"]).iloc[:100].reset_index(drop=True),
           ytepd.concat([Xte.reset_index(drop=True).iloc[:100],
           pd.DataFrame(preds, columns=["pred_energy", "pred_time"]).iloc[:100].reset.reset_index(drop=True).iloc[:100].reset_index(drop=True)],
          axis=1).to_csv("Xte_sample_pred_truth.csv", index=False)
print("Wrote sample_index(drop=True),
           yte.reset_index(drop=True).iloc[:100].reset_index(drop=True)],
          axis=1).to_csv("Xte_sample_pred_truth.csv", index=False)
print("Wrote sample diagnostics to Xte_sample_pred_truth.csv")

with open("pipeline_model.pkl", "wb") as f:
    pickle.dump(model_pipe, f, protocol=pickle.HIGHEST_PROTOCOL)

imp = model_pipe.named_steps["prep"].named_transformers diagnostics to Xte_sample_pred_truth.csv")

with open("pipeline_model.pkl", "wb") as f:
    pickle.dump(model_pipe, f, protocol=pickle.HIGHEST_PROTOCOL)

imp = model_pipe.named_steps["prep"].named_transformers_["num"]
num_means = imp.statistics_.tolist()
ohe = model_pipe.named_steps["prep"].named_transformers_["cat"]
categories = [list(c) for c in ohe.categories_]

_["num"]
num_means = imp.statistics_.tolist()
ohe = model_pipe.named_steps["prep"].named_transformers_["cat"]
categories = [list(c) for c in ohe.categories_]

def lgb_tree_to_dict(tree_struct):
    if "leaf_value" in tree_struct:
        return {"leaf": True, "value": float(tree_structdef lgb_tree_to_dict(tree_struct):
    if "leaf_value" in tree_struct:
        return {"leaf": True, "value": float(tree_struct["leaf_value"])}
    feature = int(tree_struct.get("split_feature", -1))
    threshold = float(tree_struct.get("threshold", 0.0))
    left =["leaf_value"])}
    feature = int(tree_struct.get("split_feature", -1))
    threshold = float(tree_struct.get("threshold", 0.0))
    left = lgb_tree_to_dict(tree_struct["left_child"])
    right = lgb_tree_to_dict(tree_struct lgb_tree_to_dict(tree_struct["left_child"])
    right = lgb_tree_to_dict(tree_struct["right_child"])
    return {"leaf": False, "feature": feature, "threshold": threshold, "left": left, "right": right}

def lgb_model_to_forest(estimator):
    booster = estimator.booster_
    dump = booster.dump_model()
    trees = []
    for t in dump.get("tree_info", []):
        tree_struct = t.get("tree_structure", {})
        trees.append(lgb_tree["right_child"])
    return {"leaf": False, "feature": feature, "threshold": threshold, "left": left, "right": right}

def lgb_model_to_forest(estimator):
    booster = estimator.booster_
    dump = booster.dump_model()
    trees = []
    for t in dump.get("tree_info", []):
        tree_struct = t.get("tree_structure", {})
        trees.append(lgb_tree_to_dict(tree_struct_to_dict(tree_struct))
    return trees

mor = model_pipe.named_steps["reg"]
forests = []
for est in mor.estimators_:
    try:
        trees = lgb_model_to_forest(est)
    except Exception:
        try:
            trees = lgb_model_to_forest(est)
        except Exception:
            trees))
    return trees

mor = model_pipe.named_steps["reg"]
forests = []
for est in mor.estimators_:
    try:
        trees = lgb_model_to_forest(est)
    except Exception:
        try:
            trees = lgb_model_to_forest(est)
        except Exception:
            trees = []
    forests.append(trees)

 = []
    forests.append(trees)

spec = {"cat_cols": cat_cols, "num_cols": num_cols, "num_means": num_means,
        "categories": categories, "forests": forests, "alpha": [1.0, 1.0], "beta": [0.0, 0.0]}

with open("pipeline_model_numpy.pklspec = {"cat_cols": cat_cols, "num_cols": num_cols, "num_means": num_means,
        "categories": categories, "forests": forests, "alpha": [1.0, 1.0], "beta": [0.0, 0.0]}

with open("pipeline_model_numpy.pkl", "wb") as f:
    pickle.dump(spec, f, protocol=pickle.HIGHEST_PROTOCOL)

with open("pipeline_columns.txt", "w") as f:
    for col in cat_cols + num_cols:
        f.write(col + "\n")
print("Saved pipeline_model.pkl, pipeline_model_numpy.pkl and pipeline_columns.txt (LightGBM)")

def pareto_mask(E: np.ndarray, T: np.ndarray) -> np.ndarray:
    keep = np.ones(len(E), dtype=bool)
    for i in range(len(E)):
        if not keep[i]:
            continue
        dom = (E <= E[i]) & (T <= T[i]) & ((E < E[i]) | (T < T[i]))
", "wb") as f:
    pickle.dump(spec, f, protocol=pickle.HIGHEST_PROTOCOL)

with open("pipeline_columns.txt", "w") as f:
    for col in cat_cols + num_cols:
        f.write(col + "\n")
print("Saved pipeline_model.pkl, pipeline_model_numpy.pkl and pipeline_columns.txt (LightGBM)")

def pareto_mask(E: np.ndarray, T: np.ndarray) -> np.ndarray:
    keep = np.ones(len(E), dtype=bool)
    for i in range(len(E)):
        if not keep[i]:
            continue
        dom = (E <= E[i]) & (T <= T[i]) & ((E < E[i]) | (T < T[i]))
        dom[i] =        dom[i] = False
        if dom.any():
            keep[i] = False
    return keep

def random_candidates(wt, wl, ns, N=2000):
    wl_arr = np.atleast_1d(wl)
    if wl_arr.size == 1:
        wl_arr = np.full(ns, wl_arr.item(), dtype=int)
    if ns not in grouping_plans:
        grouping_plans[ns] = generate_groupings(ns)
    rows = []
    plans False
        if dom.any():
            keep[i] = False
    return keep

def random_candidates(wt, wl, ns, N=2000):
    wl_arr = np.atleast_1d(wl)
    if wl_arr.size == 1:
        wl_arr = np.full(ns, wl_arr.item = grouping_plans[ns]
    for _ in range(N):
        gid = random.randint(1, len(plans))
        groups = plans[gid - 1]
        desc = ",".join("+".join(f"s{i}" for i in g) for g in groups)
        b, l = random.choice(core_combos)
        fs = random.choice(freq_samples)
        avg_l = float(np.mean(fs[:4])); avg_b = float(np.mean(fs[4:]))
        ths = [random.choice(threads_per_stage) for _ in groups]
        while sum(ths) > 8:
            ths = [random.choice(threads_per_stage) for _ in groups]
        qs = [random.choice(queue_sizes) for _ in groups]
        entry = {"workload_type": wt, "num_stages": ns, "grouping": gid, "group_description": desc,
                 "core_binding_type": "manual_roundrobin", "big_cores_on": b, "little_cores_on": l,
                 "total_cores_on": b + l, "imbalance": abs(b - l), "ratio_big_to_little": b / (l + 1e-6),
                 "avg_little_freq": avg_l, "avg_big_freq": avg_b}
        for s in range(1, ns + 1):
            entry[f"workload_size_stage{s}"] = int(wl_arr[s - 1])
            gi = [i for i, g in enumerate(groups) if s in g][0]
            entry[f"stage{s}_threads"] = ths[gi]
            entry[f"queue{s}_size"] = qs[gi]
        for s in range(ns + 1, 11):
            entry[f"workload_size_stage{s}"] = 0
            entry[f"stage{s}_threads"] = 0
            entry[f"queue{s}_size"] = 0
        rows.append(entry)
    mask = (df_real["workload_type"] == wt) & (df_real["num_stages"] == ns)
    for i in range(1, ns + 1):
        col = f"workload_size_stage{i}"
        ser = pd.to_numeric(df_real.get(col, pd.Series(dtype=float)), errors="coerce").fillna(-1).astype(int)
        mask &= (ser == wl_arr[i - 1])
    real_count = int(mask.sum())
    for _, r in df_real[mask].iterrows():
        cand = {k: r[k] for k in rows[0].keys() if k in r.index}
        for s in range(ns + 1, 11):
            cand[f"stage{s}_threads"] = 0
            cand[f"queue{s}_size"] = 0
        rows.append(cand)
    return pd.DataFrame(rows), real_count

def build_cli(row, wl, ns):
    b = int(row["big_cores_on"]); l = int(row["little_cores_on"])
    lf = row["avg_little_freq"] / 1e6; bf = row["avg_big_freq"] / 1e6
    grp = int(row["grouping"]); wt = row["workload_type"]
    wls = " ".join(str(int(x)) for x in wl)
    groups = grouping_plans[ns][grp - 1]
    qs = " ".join(str(int(row[f"queue{g[0]}_size"])) for g in groups)
    th = " ".join(str(int(row[f"stage{g[0]}_threads"])) for g in groups)
    return (f"--big-cores {b} --little-cores {l} --little-freq {lf:.1f}GHz --big-freq {bf:.1f}GHz "
            f"./pipe {ns} {wt} {wls} {grp} {qs} {th}")

def cli_key_from_row_pipe(row, wl, ns) -> str:
    return build_cli(row, wl, ns)

def recommend_all(wt, wl, ns, n_candidates=3000, K=5):
    cand_df, real_count = random_candidates(wt, wl, ns, n_candidates)
    if real_count == 0:
        print("No exact-match real runs; using ML for all candidates")
    Xc = pd.concat([cand_df[cat_cols].astype(str), cand_df[num_cols]], axis=1)
    pred_latency_cand_s, pred_latency_cand_std_s, preds = predict_perf(model_pipe, Xc, reps=3)
    cand_df["energy"], cand_df["time"] = preds[:, 0], preds[:, 1]
    calib = Path("calib.csv")
    if calib.exists():
        df_cal = pd.read_csv(calib)
        Xcal = pd.concat([df_cal[cat_cols].astype(str), df_cal[num_cols]], axis=1)
        pcal = model_pipe.predict(Xcal)
        lr_e = LinearRegression().fit(pcal[:, [0]], df_cal.energy)
        lr_t = LinearRegression().fit(pcal[:, [1]], df_cal.time)
        cand_df["energy"] = lr_e.predict(cand_df[["energy"]])
        cand_df["time"] = lr_t.predict(cand_df[["time"]])
        print("Applied linear calibration")
    else:
        print("calib.csv not found; skipping calibration")
    mask = pareto_mask(cand_df["energy"].to_numpy(), cand_df["time"].to_numpy())
    pf = cand_df[mask].copy()
    pf["score_balanced"] = pf.energy / pf.energy.max() + pf.time / pf.time.max()
    np.save("pipeline_configs.npy", pd.concat([pf[cat_cols].astype(str), pf[num_cols]], axis=1).to_numpy())
    print("Saved pipeline_configs.npy")
    results = {"energy": build_cli(pf.nsmallest(1, "energy").iloc[0], wl, ns),
               "time": build_cli(pf.nsmallest(1, "time").iloc[0], wl, ns),
               "balanced": build_cli(pf.nsmallest(1, "score_balanced").iloc[0], wl, ns)}
    def top_alts(df, key):
        df2 = df.copy(); best_idx = df2[key].idxmin(); alts = df2.nsmallest(K + 1, key).drop(best_idx)
        seen, uniq = set(), []
        for _, r in alts.iterrows():
            cli = build_cli(r, wl, ns)
            if cli in seen: continue
            seen.add(cli); uniq.append(cli)
            if len(uniq) == K: break
        return uniq
    alts = {"energy": top_alts(pf, "energy"), "time": top_alts(pf, "time"), "balanced": top_alts(pf, "score_balanced")}
    pf_keys = set(cli_key_from_row_pipe(r, wl, ns) for _, r in pf.iterrows())
    best_energy_cli = results["energy"]; best_time_cli = results["time"]; best_bal_cli = results["balanced"]
    def topk_set(df, key):
        s = []; best_idx = df[key].idxmin()
        for _, r in df.nsmallest(K + 1, key).drop(best_idx).iterrows():
            s.append(cli_key_from_row_pipe(r, wl, ns))
        return set(s)
    topk_energy = topk_set(pf, "energy"); topk_time = topk_set(pf, "time"); topk_bal = topk_set(pf, "score_balanced")
    metrics_row = {
        "timestamp": time.time(), "seed": SEED, "workload_type": wt, "workload_sizes": list(map(int, wl)),
        "num_stages": ns, "n_candidates": len(cand_df),
        "train_time_s": float(train_time_s), "peak_mem_bytes": int(peak_mem_bytes), "model_size_bytes": int(model_size_bytes),
        "pred_latency_per_sample_ms_candidates": float(pred_latency_cand_s * 1e3),
        "pred_latency_std_ms_candidates": float(pred_latency_cand_std_s * 1e3),
        "pareto_count": int(len(pf_keys)),
        "best_energy_cli": best_energy_cli, "best_time_cli": best_time_cli, "best_balanced_cli": best_bal_cli,
        "topk_energy": sorted(list(topk_energy)), "topk_time": sorted(list(topk_time)), "topk_balanced": sorted(list(topk_bal)),
        "baseline_mae_energy": baseline_mae_e, "baseline_mae_time": baseline_mae_t,
        "yte_std_energy": yte_std_energy, "yte_std_time": yte_std_time, "r2_reliable": bool(r2_reliable)
    }
    append_metrics(metrics_row)
    artifacts = {"pf_keys": pf_keys, "best": {"energy": best_energy_cli, "time": best_time_cli, "balanced": best_bal_cli},
                 "topk": {"energy": topk_energy, "time": topk_time, "balanced": topk_bal}}
    return results, alts, artifacts

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train LightGBM surrogate & emit Pareto configs")
    parser.add_argument("--n_candidates", "-c", type=int, default=3000)
    parser.add_argument("--K", "-k", type=int, default=5)
    args, _ = parser.parse_known_args()

    wts = ["cpu_only", "memory_only", "cpu_and_memory", "memory_and_cpu"]
    vectors = {
        "cpu_only": [[1_000_000] * 3, [5_000_000] * 3, [10_000_000] * 3, [1_000_000, 2_000_000, 3_000_000]],
        "memory_only": [[2_000_000] * 4, [4_000_000] * 4],
        "cpu_and_memory": [[8_000_000] * 2, [16_000_000] * 2],
        "memory_and_cpu": [[8_000_000] * 2, [16_000_000] * 2],
    }

    all_pf = []
    rmse_energy_list = []; rmse_time_list = []
    mae_energy_list = []; mae_time_list = []
    r2_energy_list = []; r2_time_list = []
    consolidated_best_clis = []; consolidated_alt_clis = set()
    combos_processed = 0

    for wt in wts:
        for wl in vectors.get(wt, []):
            ns = len(wl)
            results, alts, artifacts = recommend_all(wt=wt, wl=wl, ns=ns,
                                                    n_candidates=args.n_candidates, K=args.K)
            wlstr = "_".join(str(x) for x in wl)
            combo = f"{wt}_{ns}stages_{wlstr}"
            print(f"\n=== RESULTS for {combo} ===")
            print("ENERGY   →", results["energy"])
            print("TIME     →", results["time"])
            print("BALANCED →", results["balanced"])
            print(f"\n--- ALTERNATIVES for {combo} ---")
            for obj in ("balanced", "energy", "time"):
                for cli in alts[obj]:
                    print(" ", cli)
            pf_arr = np.load("pipeline_configs.npy", allow_pickle=True)
            all_pf.append(pf_arr)
            consolidated_best_clis.append({"combo": combo, "best_energy": artifacts["best"]["energy"],
                                           "best_time": artifacts["best"]["time"], "best_balanced": artifacts["best"]["balanced"]})
            for s in artifacts["pf_keys"]:
                consolidated_alt_clis.add(s)
            for objset in artifacts["topk"].values():
                for k in objset:
                    consolidated_alt_clis.add(k)
            rmse_energy_list.append(float(rmse_e)); rmse_time_list.append(float(rmse_t))
            mae_energy_list.append(float(mae_e)); mae_time_list.append(float(mae_t))
            r2_energy_list.append(float(r2_e)); r2_time_list.append(float(r2_t))
            combos_processed += 1
            print(f"Processed {combo}")

    combined = np.vstack(all_pf) if all_pf else np.zeros((0, len(cat_cols) + len(num_cols)))
    np.save(CONSOLIDATED_PF, combined)
    with open(CONSOLIDATED_BEST, "w") as f:
        for entry in consolidated_best_clis:
            f.write(json.dumps(entry) + "\n")
    with open(CONSOLIDATED_ALT, "w") as f:
        for cli in sorted(consolidated_alt_clis):
            f.write(cli + "\n")

    def summarize(lst):
        if not lst: return {"mean": None, "std": None, "count": 0}
        a = np.array(lst, dtype=float); return {"mean": float(a.mean()), "std": float(a.std(ddof=0)), "count": int(a.size)}

    totals = {
        "rmse_energy": summarize(rmse_energy_list),
        "rmse_time": summarize(rmse_time_list),
        "mae_energy": summarize(mae_energy_list),
        "mae_time": summarize(mae_time_list),
        "r2_energy": summarize(r2_energy_list),
        "r2_time": summarize(r2_time_list),
        "combos_processed": combos_processed
    }

    rmse_total = totals["rmse_energy"]["mean"] + totals["rmse_time"]["mean"] if totals["rmse_energy"]["mean"] is not None else None
    mae_total = totals["mae_energy"]["mean"] + totals["mae_time"]["mean"] if totals["mae_energy"]["mean"] is not None else None
    r2_total = totals["r2_energy"]["mean"] + totals["r2_time"]["mean"] if totals["r2_energy"]["mean"] is not None else None

    print("\n=== AGGREGATED METRICS ACROSS COMBOS (TOTAL) ===")
    print(f"RMSE energy mean={totals['rmse_energy']['mean']:.3f} std={totals['rmse_energy']['std']:.3f}")
    print(f"RMSE time   mean={totals['rmse_time']['mean']:.3f} std={totals['rmse_time']['std']:.3f}")
    print(f"MAE  energy mean={totals['mae_energy']['mean']:.3f} std={totals['mae_energy']['std']:.3f}")
    print(f"MAE  time   mean={totals['mae_time']['mean']:.3f} std={totals['mae_time']['std']:.3f}")
    print(f"R2   energy mean={totals['r2_energy']['mean']:.6f} std={totals['r2_energy']['std']:.6f}")
    print(f"R2   time   mean={totals['r2_time']['mean']:.6f} std={totals['r2_time']['std']:.6f}")

    print("\n=== EXPLICIT TOTALS (SUM OF ENERGY + TIME MEANS) ===")
    print(f"RMSE total = RMSE_energy_mean + RMSE_time_mean = {rmse_total:.3f}")
    print(f"MAE  total = MAE_energy_mean  + MAE_time_mean  = {mae_total:.3f}")
    print(f"R2   total = R2_energy_mean   + R2_time_mean   = {r2_total:.6f}")

    summary_out = {
        "totals": totals,
        "explicit_totals": {"rmse_total": rmse_total, "mae_total": mae_total, "r2_total": r2_total},
        "global_test": {"rmse_energy": float(rmse_e), "rmse_time": float(rmse_t),
                        "mae_energy": float(mae_e), "mae_time": float(mae_t),
                        "r2_energy": float(r2_e), "r2_time": float(r2_t)},
        "model": {"n_estimators": lgb_params["n_estimators"], "model_size_bytes": model_size_bytes,
                  "train_time_s": train_time_s, "peak_mem_bytes": peak_mem_bytes},
        "notes": "Totals are means across combos; explicit_totals are sums of energy+time means."
    }
    with open("pipeline_aggregated_summary.json", "w") as f:
        json.dump(summary_out, f, indent=2)

    print(f"\nSaved COMBINED {CONSOLIDATED_PF} ({combined.shape[0]} total configs across all WT×WL)")
    print(f"Wrote consolidated best CLIs → {CONSOLIDATED_BEST}")
    print(f"Wrote consolidated alt CLIs  → {CONSOLIDATED_ALT}")
    print(f"Wrote aggregated summary     → pipeline_aggregated_summary.json")
    print(f"Per-combo metrics log        → {METRICS_LOG}")
